In [ ]:
# Instalação (Google Colab)
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium beautifulsoup4 > /dev/null

# Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import time, os, re, requests, zipfile
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

# Setup
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_imagens = "imagens_telhanorte"
os.makedirs(pasta_imagens, exist_ok=True)

# Configuração do navegador
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# Coletar todos os links de produtos (várias páginas)
base_url = "https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page="
links = []
pagina = 1

print("🔎 Coletando links de produtos...")
while True:
    driver.get(base_url + str(pagina))
    time.sleep(3)
    produtos = driver.find_elements(By.CSS_SELECTOR, "a.vtex-product-summary-2-x-clearLink")
    novos_links = [p.get_attribute("href") for p in produtos if p.get_attribute("href") not in links]

    if not novos_links:
        print(f"✅ Fim da coleta na página {pagina}")
        break

    links.extend(novos_links)
    print(f"🔗 Página {pagina}: {len(novos_links)} novos links")
    pagina += 1

driver.quit()

# Função para extrair dados
def extrair_detalhes_produto(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        nome = soup.find("h1", class_="vtex-store-components-3-x-productNameContainer")
        nome = nome.text.strip() if nome else "N/A"

        codigo = soup.find("span", class_="vtex-product-identifier-0-x-product-identifier__value")
        codigo = codigo.text.strip() if codigo else "0"

        preco_original = soup.select_one(".vtex-product-price-1-x-listPriceWithUnitMultiplier strike")
        preco_original = preco_original.text.strip() if preco_original else "N/A"

        preco_promocional = soup.select_one(".telhanorte-telha-store-app-1-x-preco-best-un")
        preco_promocional = preco_promocional.text.strip() if preco_promocional else "N/A"

        preco_m2 = soup.select_one(".telhanorte-telha-store-app-1-x-preco-m2")
        preco_m2 = preco_m2.text.strip() if preco_m2 else "N/A"

        imagem = soup.find("img", class_="telhanorte-telha-store-app-1-x-imagem-destaque")
        imagem_url = imagem['src'] if imagem else "N/A"

        return [nome, codigo, preco_original, preco_promocional, preco_m2, imagem_url, url]
    except Exception as e:
        print(f"Erro em {url}: {e}")
        return None

# Função para salvar imagem
def salvar_imagem(url_img, nome_base):
    if url_img == "N/A": return
    try:
        resposta = requests.get(url_img, timeout=10)
        if resposta.status_code == 200:
            caminho = os.path.join(pasta_imagens, f"{nome_base}.jpg")
            with open(caminho, 'wb') as f:
                f.write(resposta.content)
    except Exception as e:
        print(f"Erro ao baixar imagem: {e}")

# Coletar os dados
dados_produtos = []
for i, link in enumerate(links):
    print(f"[{i+1}/{len(links)}] Coletando dados: {link}")
    resultado = extrair_detalhes_produto(link)
    if resultado:
        dados_produtos.append(resultado)
        # salvar imagem com código como nome
        codigo_produto = re.sub(r'[\\/*?:"<>|]', "", resultado[1])
        if not codigo_produto or codigo_produto == "0":
            codigo_produto = re.sub(r'[\\/*?:"<>|]', "", resultado[0])[:50]
        salvar_imagem(resultado[5], codigo_produto)

# Salvar CSV com separador | e encoding compatível com Excel
df = pd.DataFrame(dados_produtos, columns=[
    "Nome", "Código", "Preço Original", "Preço Promocional", "Preço por m²", "Imagem", "URL"
])
csv_path = f"telhanorte_{data_hoje}.csv"
df.to_csv(csv_path, sep='|', index=False, encoding='utf-8-sig')
print(f"\n📄 CSV salvo: {csv_path}")

# Compactar imagens
zip_path = f"imagens_telhanorte_{data_hoje}.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(pasta_imagens):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"📦 Imagens compactadas em: {zip_path}")
